In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

import XPOSim as xpos


# ============================================================
# 1. SIMULATION GRID
# ============================================================

DX = 5e-8
DY = 5e-8

NX = 2**11
NY = 2**11

# Размер физического расчетного окна:
# X_ROI, Y_ROI = 30e-6, 30e-6   # SNIGIREV
X_ROI, Y_ROI = 20e-6, 20e-6      # FTIAN

grid = xpos.Grid2D(
    dx=DX,
    dy=DY,
    nx=NX,
    ny=NY,
    backend="cupy",
)

print(f"поле X: ± {NX * DX * 1e6 / 2:.3f} мкм")
print(f"поле Y: ± {NY * DY * 1e6 / 2:.3f} мкм")


# Координаты для визуализации
x = cp.asnumpy(grid.x_axis) * 1e6
y = cp.asnumpy(grid.y_axis) * 1e6

поле X: ± 51.200 мкм
поле Y: ± 51.200 мкм


In [3]:
# ============================================================
# 2. X-RAY SOURCE
# ============================================================

SOURCE_DISTANCE = 15.0       # m
ENERGY = 12.0                # keV

source = xpos.PointSource(
    grid=grid,
    position=(0.0, 0.0),
    energy_kev=ENERGY,
)

# Волновая функция в плоскости первой CRL
E_source = source.E(z=SOURCE_DISTANCE)


# ============================================================
# 3. CRL PARAMETERS
# ============================================================

N_LENSES = 26

RADIUS = 6.25e-6             # m
APERTURE = 50e-6              # m
MINIMUM_THICKNESS = 2e-6      # m

TRANSVERSE_LENGTH = 33e-6     # FTIAN2
# TRANSVERSE_LENGTH = 53e-6   # SNIGIREV

material = xpos.Material(
    formula="Si",
    density=2.33,
)


# ============================================================
# 4. REFERENCE CRL
# ============================================================

# Старый CRL1D(..., hor=False):
#
#   - линза фокусирует по Y
#   - поперечная длина ограничивает X
#
# В новой геометрии это можно задать поворотом CRL
# на +90 градусов относительно локальной системы.

crl_reference = xpos.CRL(
    input_field=E_source,
    grid=grid,
    radius=RADIUS,
    aperture=APERTURE,
    minimum_thickness=MINIMUM_THICKNESS,
    n_lenses=N_LENSES,
    material=material,
    focusing_axis="x",
    transverse_length=TRANSVERSE_LENGTH,
    rotation=(0.0, 0.0, np.pi / 2),
)


theoretical_focus = crl_reference.theoretical_focus()

print(
    f"теоретическое фокусное расстояние = "
    f"{theoretical_focus * 1e3:.3f} мм"
)

print(
    f"характеристическая длина CRL = "
    f"{crl_reference.characteristic_length * 1e3:.3f} мм"
)

print(
    f"критическое число линз = "
    f"{crl_reference.critical_lens_count:.2f}"
)

if not crl_reference.is_focus_outside_crl():
    print(
        "ВНИМАНИЕ: теоретический фокус находится внутри CRL; "
        "внешний фокусный анализ физически неприменим."
    )


# ============================================================
# 5. TWO-CRL SCHEME
# ============================================================

N1 = 6
N2 = 40

# Другие старые варианты:
# N1, N2 = 6, 58
# N1, N2 = 14, 104
# N1, N2 = 14, 132


# ------------------------------------------------------------
# First CRL
# ------------------------------------------------------------

# Первая CRL фокусирует по Y.

crl_1 = xpos.CRL(
    input_field=E_source,
    grid=grid,
    radius=RADIUS,
    aperture=APERTURE,
    minimum_thickness=MINIMUM_THICKNESS,
    n_lenses=N1,
    material=material,
    focusing_axis="x",
    transverse_length=TRANSVERSE_LENGTH,
    rotation=(0.0, 0.0, np.pi / 2),
)


# ------------------------------------------------------------
# Second CRL
# ------------------------------------------------------------

# Чтобы рассчитать положение второй CRL, нам нужен ее
# аналитический effective focus.
#
# В текущем API effective_focus является методом конкретной
# CRL и использует ее число линз, поэтому для N2 создаем
# отдельную конфигурацию аналитической CRL.
#
# Поле для нее считать не нужно.

crl_2_geometry = xpos.CRL(
    input_field=E_source,
    grid=grid,
    radius=RADIUS,
    aperture=APERTURE,
    minimum_thickness=MINIMUM_THICKNESS,
    n_lenses=N2,
    material=material,
    focusing_axis="x",
    transverse_length=TRANSVERSE_LENGTH,
    rotation=(0.0, 0.0, 0.0),
)


# ------------------------------------------------------------
# Distance between CRLs
# ------------------------------------------------------------

# Pitch одного элемента CRL
pitch = crl_1.lens_pitch

focus_1 = crl_1.effective_focus(
    source_distance=SOURCE_DISTANCE,
)

focus_2 = crl_2_geometry.effective_focus(
    source_distance=SOURCE_DISTANCE,
)

Z1 = focus_1 - focus_2 - N2 * pitch

print(
    f"расстояние между CRL = "
    f"{Z1 * 1e3:.3f} мм"
)


# ------------------------------------------------------------
# Propagation from CRL 1 to CRL 2
# ------------------------------------------------------------

# E(z=0) -> поле непосредственно после CRL 1.
# E(z=Z1) -> то же поле после распространения на Z1.

E_before_crl_2 = crl_1.E(z=Z1)


# ============================================================
# 6. SECOND CRL
# ============================================================

# Аналог старого:
#
#     CRL1D(..., hor=True)
#
# То есть теперь линза фокусирует по X.

crl_2 = xpos.CRL(
    input_field=E_before_crl_2,
    grid=grid,
    radius=RADIUS,
    aperture=APERTURE,
    minimum_thickness=MINIMUM_THICKNESS,
    n_lenses=N2,
    material=material,
    focusing_axis="x",
    transverse_length=TRANSVERSE_LENGTH,
    rotation=(0.0, 0.0, 0.0),
)


# ============================================================
# 7. IMAGE PLANE
# ============================================================

# В старом коде effective_focus использовался с SOURCE_DISTANCE.
# Для новой схемы здесь нужно использовать фактическое
# расстояние от источника до второй CRL, если рассчитывается
# физическая object-image геометрия всей схемы.
#
# Поэтому:
SECOND_CRL_OBJECT_DISTANCE = SOURCE_DISTANCE + Z1

Z2 = crl_2.effective_focus(
    source_distance=SECOND_CRL_OBJECT_DISTANCE,
)

print(
    f"расстояние от второй CRL до изображения = "
    f"{Z2 * 1e3:.3f} мм"
)


# ============================================================
# 8. FIELD IN IMAGE PLANE
# ============================================================

E_image = crl_2.E(z=Z2)

I_image = xpos.intensity(E_image)


# ============================================================
# 9. BASIC BEAM ANALYSIS
# ============================================================

metrics = xpos.beam_metrics(
    E_image,
    grid,
    roi=(
        (-1e-6, 1e-6),
        (-1e-6, 1e-6),
    ),
    calculate_divergence=False,
    k=crl_2.k,
)

print(f"максимальная интенсивность = {metrics.max_intensity:.6g}")
print(f"центр пучка = {metrics.center}")
print(f"FWHM = {metrics.fwhm}")


# ============================================================
# 10. VISUALIZATION
# ============================================================

plt.figure(figsize=(7, 6))

plt.pcolormesh(
    x,
    y,
    cp.asnumpy(I_image),
    shading="auto",
)

plt.xlim(-1, 1)
plt.ylim(-1, 1)

plt.xlabel("x, мкм")
plt.ylabel("y, мкм")

plt.title(
    f"Intensity in image plane, z = {Z2 * 1e3:.3f} mm"
)

plt.colorbar(label="Intensity")

plt.tight_layout()
plt.show()

ValueError: Specify either wavelength_m or energy_kev.